# Jena Climate: Reproducible Deep Learning Comparison

Compare FFNN, Vanilla RNN, LSTM, and GRU models for 4-hour-ahead temperature forecasting. The notebook uses the reusable modules in `src/` and fixed random seeds.

In [ ]:
from pathlib import Path
import os, random, sys
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import build_sequence_model
from src.preprocessing import create_sequences
from src.evaluation import regression_metrics, inverse_transform_target
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = PROJECT_ROOT / "data/jena_climate_2009_2016.csv"
TARGET = "T (degC)"
INPUT_WIDTH, HORIZON = 72, 24
print("TensorFlow:", tf.__version__)

## Data and chronological split

The scaling parameters are fitted on the training split only. Windows are constructed separately within train, validation, and test splits to preserve the original assignment setup.

In [ ]:
data = pd.read_csv(DATA_PATH)
data["Date Time"] = pd.to_datetime(data["Date Time"], format="%d.%m.%Y %H:%M:%S")
data = data.set_index("Date Time").sort_index().ffill().bfill()

n = len(data)
train_end, val_end = int(.70*n), int(.90*n)
train, val, test = data.iloc[:train_end], data.iloc[train_end:val_end], data.iloc[val_end:]

mean, std = train.mean(), train.std()
train_n, val_n, test_n = (train-mean)/std, (val-mean)/std, (test-mean)/std
target_mean, target_std = float(mean[TARGET]), float(std[TARGET])

X_train, y_train = create_sequences(train_n.to_numpy(), data.columns.get_loc(TARGET), INPUT_WIDTH, HORIZON)
X_val, y_val = create_sequences(val_n.to_numpy(), data.columns.get_loc(TARGET), INPUT_WIDTH, HORIZON)
X_test, y_test = create_sequences(test_n.to_numpy(), data.columns.get_loc(TARGET), INPUT_WIDTH, HORIZON)

print(X_train.shape, X_val.shape, X_test.shape)

## Train and evaluate

All sequence architectures use the same training schedule: Adam (`1e-3`), batch size 256, 20 epochs, and seed 42.

In [ ]:
results = []
for name in ["ffnn", "rnn", "lstm", "gru"]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = build_sequence_model(name, X_train.shape[1:], learning_rate=1e-3)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=20, batch_size=256, shuffle=True, verbose=0)
    val_pred, test_pred = model.predict(X_val, verbose=0).ravel(), model.predict(X_test, verbose=0).ravel()
    yv_c, yt_c = inverse_transform_target(y_val, target_mean, target_std), inverse_transform_target(y_test, target_mean, target_std)
    vp_c, tp_c = inverse_transform_target(val_pred, target_mean, target_std), inverse_transform_target(test_pred, target_mean, target_std)
    vm, tm = regression_metrics(yv_c, vp_c), regression_metrics(yt_c, tp_c)
    results.append({"Model": {"ffnn":"FFNN","rnn":"Vanilla RNN","lstm":"LSTM","gru":"GRU"}[name],
                    "Val MAE (°C)": vm["MAE"], "Val RMSE (°C)": vm["RMSE"],
                    "Test MAE (°C)": tm["MAE"], "Test RMSE (°C)": tm["RMSE"]})

results = pd.DataFrame(results)
display(results.round(3))

In [ ]:
results.to_csv(PROJECT_ROOT / "results/tables/time_series_model_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(results["Model"], results["Test MAE (°C)"])
ax.set_ylabel("Test MAE (°C)")
ax.set_title("Jena Climate — Test MAE")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/figures/time_series_test_mae.svg")
plt.show()

## Reproducibility note

Exact neural-network metrics can vary across hardware and TensorFlow versions. This notebook controls Python, NumPy, and TensorFlow seeds and requests deterministic TensorFlow operations where supported.